# Task 1 — Data Analysis & Preprocessing

**Fraud detection for e-commerce and bank transactions (Adey Innovations Inc.)**

This notebook covers the Interim-1 deliverables:

1. Data loading & cleaning (missing values, duplicates, dtypes)
2. Exploratory Data Analysis (univariate & bivariate)
3. Geolocation: mapping IP addresses to countries
4. Feature engineering (`time_since_signup`, time-based, transaction velocity)
5. Encoding / scaling and the class-imbalance handling strategy

> Place `Fraud_Data.csv`, `IpAddress_to_Country.csv`, and `creditcard.csv` in
> `data/raw/` before running. Reusable logic lives in the `src/` package so the
> notebook stays a thin, readable driver.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable so `from src import ...` works.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data_loader, preprocessing, geolocation, feature_engineering, eda, transform
from src import config

pd.set_option("display.max_columns", 50)
print("Project root:", PROJECT_ROOT)

## 1. Load the data

In [ ]:
fraud = data_loader.load_fraud_data()
ip = data_loader.load_ip_country()
cc = data_loader.load_creditcard()

print("Fraud_Data:", fraud.shape)
print("IpAddress_to_Country:", ip.shape)
print("creditcard:", cc.shape)
fraud.head()

In [ ]:
fraud.info()
fraud.describe(include="all").T

## 2. Cleaning — missing values, duplicates, dtypes

In [ ]:
# Missing-value overview for both datasets
print("=== Fraud_Data missing values ===")
display(preprocessing.missing_value_summary(fraud))
print("=== creditcard missing values ===")
display(preprocessing.missing_value_summary(cc).head())

In [ ]:
# De-duplicate then impute. Median for numeric, mode for categorical.
fraud = preprocessing.basic_clean(fraud)
fraud = preprocessing.handle_missing_values(fraud)

cc = preprocessing.basic_clean(cc)
cc = preprocessing.handle_missing_values(cc)

print("After cleaning -> Fraud_Data:", fraud.shape, "| creditcard:", cc.shape)

## 3. Exploratory Data Analysis

### 3.1 Class imbalance (target distribution)

In [ ]:
eda.class_balance(fraud, "class", name="fraud_class_balance.png")
eda.class_balance(cc, "Class", name="creditcard_class_balance.png")
plt.show()

### 3.2 Univariate & bivariate analysis

In [ ]:
# Purchase value and age, split by fraud class
eda.numeric_distribution(fraud, "purchase_value", target="class",
                         name="purchase_value_by_class.png")
eda.numeric_distribution(fraud, "age", target="class", name="age_by_class.png")
plt.show()

In [ ]:
# Fraud rate by categorical drivers
for col in ["source", "browser", "sex"]:
    eda.fraud_rate_by_category(fraud, col, "class", name=f"fraud_rate_{col}.png")
plt.show()

## 4. Geolocation — map IP addresses to countries

Each transaction's numeric `ip_address` is matched against the
`[lower_bound, upper_bound]` ranges in `IpAddress_to_Country.csv` using a
sorted `merge_asof`, then validated against the upper bound. IPs outside every
range are labelled `Unknown`.

In [ ]:
fraud = geolocation.merge_ip_to_country(fraud, ip)
print("Countries mapped (top 10 by volume):")
display(fraud["country"].value_counts().head(10))

# Which countries carry the highest fraud rate (with enough volume)?
country_stats = (fraud.groupby("country")["class"]
                 .agg(["mean", "count"]).query("count >= 100")
                 .sort_values("mean", ascending=False).head(10))
country_stats.rename(columns={"mean": "fraud_rate", "count": "n_txn"})

## 5. Feature engineering

- **`time_since_signup`** — hours between signup and purchase. Fraudulent
  accounts frequently transact within minutes of creation.
- **Time-based** — `purchase_hour`, `purchase_dayofweek`.
- **Velocity / frequency** — `user_txn_count`, `device_txn_count`,
  `device_shared` (multiple users on one device = fraud-ring signal).

In [ ]:
fraud = feature_engineering.build_features(fraud)
new_cols = ["time_since_signup", "purchase_hour", "purchase_dayofweek",
            "user_txn_count", "device_txn_count", "device_shared"]
fraud[new_cols].describe().T

In [ ]:
# time_since_signup is one of the strongest signals — inspect it by class.
eda.numeric_distribution(fraud, "time_since_signup", target="class", log=True,
                         name="time_since_signup_by_class.png")
plt.show()
fraud.groupby("class")["time_since_signup"].median()

## 6. Encoding, scaling & class-imbalance strategy

- **Encoding:** one-hot for `source`, `browser`, `sex`, `country`
  (`min_frequency` collapses rare countries; `handle_unknown="ignore"` keeps
  inference robust).
- **Scaling:** `StandardScaler` on numeric features so scale-sensitive models
  (Logistic Regression) behave well.
- **Leakage guard:** the transformer is *fit on the training split only* in
  Task 2.
- **Class imbalance:** evaluate with AUC-PR / F1 (not accuracy) and apply
  **SMOTE on the training fold only**. Tree models additionally use
  class weights. The strategy is implemented in Task 2 to avoid leaking
  resampled rows into validation.

In [ ]:
num_cols, cat_cols = transform.split_feature_types(fraud, target="class")
print("Numeric features:", num_cols)
print("Categorical features:", cat_cols)
preprocessor = transform.build_preprocessor(num_cols, cat_cols)
preprocessor

## 7. Persist the processed datasets

In [ ]:
fraud.to_csv(config.PROCESSED_DIR / "fraud_processed.csv", index=False)
cc.to_csv(config.PROCESSED_DIR / "creditcard_processed.csv", index=False)
print("Saved processed datasets to data/processed/.")
print("Fraud_Data:", fraud.shape, "| creditcard:", cc.shape)